In this notebook, we will compare the results of our trained models

In [3]:
import os
import time
import numpy as np
import pandas as pd
from pathlib import Path

import warnings
warnings.filterwarnings('ignore', category=np.exceptions.VisibleDeprecationWarning)

## Dataset

In [4]:
import kagglehub
# kagglehub.login()

path = kagglehub.dataset_download("qingyi/wm811k-wafer-map")
print(path)

/Users/purav/.cache/kagglehub/datasets/qingyi/wm811k-wafer-map/versions/1


In [5]:
print("Loading dataset...")

file_path = "/Users/purav/.cache/kagglehub/datasets/qingyi/wm811k-wafer-map/versions/1/LSWMD.pkl" # Change this to where the LSWMD.pkl is downloaded
# file_path = "../data/LSWMD.pkl"

df_original = pd.read_pickle(file_path)
print(f"Done. {len(df_original)} rows loaded.")

Loading dataset...
Done. 811457 rows loaded.


### Dataset Cleaning

In [6]:
# 1. Rename "trianTestLabel" to "trainTestLabel"
df_original = df_original.rename(columns={"trianTestLabel":"trainTestLabel"})

In [7]:
# 2. Drop samples with unlabeled failureType and unlabeled trainTestLabel

df_original["failureType_str"] = df_original["failureType"].astype(str)
df_original["trainTestLabel_str"] = df_original["trainTestLabel"].astype(str)

In [8]:
# df_labeled contains only labeled samples
df_labeled = df_original[
    (df_original["failureType_str"] != "[]") &
    (df_original["trainTestLabel_str"] != "[]")
].copy()

df_labeled[["failureType", "trainTestLabel"]] = (
    df_labeled[["failureType_str", "trainTestLabel_str"]]
    .apply(lambda s: s.str.replace(r"[\[\]']", "", regex=True).str.strip())
)
df_labeled = df_labeled.reset_index(drop=True)

# df_unlabeled contains only unlabeled samples
df_unlabeled = df_original[
    (df_original["failureType_str"] == "[]") |
    (df_original["trainTestLabel_str"] == "[]")
].copy().reset_index(drop=True)

# df_all contains both labeled and unlabeled samples
df_all = df_original.copy().reset_index(drop=True)

In [9]:
print(f"df_all: {len(df_all):,} total samples")
print(f"  labeled: {len(df_labeled):,}")
print(f"  unlabeled: {len(df_unlabeled):,}")

print(f"\ndf_unlabeled: {len(df_unlabeled):,} samples")

print(f"\ndf_labeled: {len(df_labeled):,} samples")
print(df_labeled["failureType"].value_counts())

df_all: 811,457 total samples
  labeled: 172,950
  unlabeled: 638,507

df_unlabeled: 638,507 samples

df_labeled: 172,950 samples
failureType
none         147431
Edge-Ring      9680
Edge-Loc       5189
Center         4294
Loc            3593
Scratch        1193
Random          866
Donut           555
Near-full       149
Name: count, dtype: int64


### Preprocessing

In [10]:
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [11]:
CLASSES = ["none", "Edge-Ring", "Edge-Loc", "Center", "Loc", "Scratch", "Random", "Donut", "Near-full"]
LABEL_MAP = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_LABEL = {i: c for c, i in LABEL_MAP.items()}
NUM_CLASSES = len(CLASSES)
print(LABEL_MAP)

{'none': 0, 'Edge-Ring': 1, 'Edge-Loc': 2, 'Center': 3, 'Loc': 4, 'Scratch': 5, 'Random': 6, 'Donut': 7, 'Near-full': 8}


Resize and pad function

In [12]:
def resize_and_pad(wm, target=64):
    """
    Resize while maintaining aspect ratio.
    Resize longest side to target, pad shorter side with zeros (0 = outside wafer).
    """
    h, w  = wm.shape
    scale = target / max(h, w)
    new_h, new_w = int(h * scale), int(w * scale)

    wm_img = Image.fromarray(wm).resize((new_w, new_h), Image.NEAREST)

    pad_h = target - new_h
    pad_w = target - new_w
    wm_padded = np.pad(
        np.array(wm_img),
        ((pad_h // 2, pad_h - pad_h // 2),
         (pad_w // 2, pad_w - pad_w // 2)),
        mode="constant",
        constant_values=0
    )
    return wm_padded

In [13]:
class WaferDataset(Dataset):
    def __init__(self, df, strategy="pad", return_label=False):
        """
        Args:
            df: dataframe with at least a waferMap column
            strategy: "pad" to preserve aspect ratio with padding,
                      "resize" to stretch directly to IMG_SIZE x IMG_SIZE
            return_label: if True, also return the mapped label
        """
        self.df = df.reset_index(drop=True)
        self.strategy = strategy
        self.return_label = return_label

    def __len__(self):
        return len(self.df)

    def _shape_normalize(self, wm):
        if self.strategy == "pad":
            return resize_and_pad(wm)
        elif self.strategy == "resize":
            return np.array(
                Image.fromarray(wm).resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)
            )
        else:
            raise ValueError(f"Unknown strategy: {self.strategy}")

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Convert wafer map to float and scale values to [0, 1]
        wm = np.array(row["waferMap"], dtype=np.float32) / 2.0

        # Apply preprocessing only
        wm_array = self._shape_normalize(wm)
        wm_tensor = torch.from_numpy(wm_array).unsqueeze(0)  # (1, H, W)

        if self.return_label:
            label = LABEL_MAP[row["failureType"]]
            return wm_tensor, label

        return wm_tensor

In [14]:
# Config

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

BATCH_SIZE = 256
# NUM_WORKERS = 8 # number of cores / 2
NUM_WORKERS = 0

STRATEGY = "pad"  # Choose the normalization strategy ("resize" or "pad")

VAL_SPLIT  = 0.1 # % of dataset used as validation set
TEST_SPLIT = 0.2 # % of dataset used as test set

In [15]:
df_train_val, df_test = train_test_split(
    df_labeled,
    test_size=TEST_SPLIT,
    random_state=SEED,
    stratify=df_labeled["failureType"]
)

df_train_final, df_val = train_test_split(
    df_train_val,
    test_size=VAL_SPLIT / (1 - TEST_SPLIT),
    random_state=SEED,
    stratify=df_train_val["failureType"]
)

In [16]:
def make_labeled_loader(df, batch_size, strategy="pad", shuffle=False, num_workers=8):
    dataset = WaferDataset(df, strategy=strategy, return_label=True)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
    )

def make_unlabeled_loader(df, batch_size, strategy="pad", shuffle=False, num_workers=8):
    dataset = WaferDataset(df, strategy=strategy, return_label=False)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
    )

In [17]:
test_loader = make_labeled_loader(
    df_test,
    batch_size=BATCH_SIZE,
    strategy=STRATEGY,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

unlabeled_loader = make_unlabeled_loader(
    df_unlabeled,
    batch_size=BATCH_SIZE,
    strategy=STRATEGY,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

In [18]:
def get_best_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    elif torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

device = get_best_device()
print(f"Device: {device}")

Device: mps


## Results

In [19]:
import torch.nn as nn
from tqdm.notebook import tqdm
from sklearn.metrics import classification_report, confusion_matrix

In [20]:
def evaluate_classifier(
    model,
    loader,
    class_names,
    device,
    run_name="model",
    label_smoothing=0.0,
    training_metadata=None,
):
    model.eval()
    all_preds, all_labels = [], []
    all_confs = []
    running_loss = 0.0
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    with torch.no_grad():
        for x, y in tqdm(loader, desc=f"Evaluating {run_name}"):
            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            probs = torch.softmax(logits, dim=1)
            confs, preds = probs.max(dim=1)

            loss = criterion(logits, y)
            running_loss += loss.item()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            all_confs.extend(confs.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_confs = np.array(all_confs)

    test_loss = running_loss / len(loader)
    report = classification_report(
        all_labels,
        all_preds,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
    cm = confusion_matrix(all_labels, all_preds)

    print(f"\n--- {run_name} ---")
    print(f"Test Loss: {test_loss:.4f}")
    print(f"\nPer-class Metrics:")
    print(f"{'Class':<12} {'F1':>10} {'Precision':>10} {'Recall':>10} {'Mean Conf':>12}")
    print("-" * 60)

    for class_idx, cls in enumerate(class_names):
        cls_mask = all_preds == class_idx
        mean_conf = all_confs[cls_mask].mean() if cls_mask.any() else float("nan")

        print(
            f"{cls:<12} "
            f"{report[cls]['f1-score']:>10.4f} "
            f"{report[cls]['precision']:>10.4f} "
            f"{report[cls]['recall']:>10.4f} "
            f"{mean_conf:>12.4f}"
        )

    print("-" * 60)
    print(
        f"{'macro avg':<12} "
        f"{report['macro avg']['f1-score']:>10.4f} "
        f"{report['macro avg']['precision']:>10.4f} "
        f"{report['macro avg']['recall']:>10.4f} "
        f"{all_confs.mean():>12.4f}"
    )
    print(f"\nOverall Accuracy: {report['accuracy']:.4f}")

    if training_metadata is not None:
        if "training_time_sec" in training_metadata:
            print(f"Training Time: {training_metadata['training_time_sec'] / 60:.4f} min")
        if "best_val_loss_at_f1" in training_metadata:
            print(f"Best Val Loss at F1: {training_metadata['best_val_loss_at_f1']:.4f}")
        if "best_train_loss_at_f1" in training_metadata:
            print(f"Best Train Loss at F1: {training_metadata['best_train_loss_at_f1']:.4f}")

    return {
        "test_loss": round(test_loss, 4),
        "accuracy": round(report["accuracy"], 4),
        "macro_f1": round(report["macro avg"]["f1-score"], 4),
        "macro_precision": round(report["macro avg"]["precision"], 4),
        "macro_recall": round(report["macro avg"]["recall"], 4),
        "mean_confidence": round(float(all_confs.mean()), 4),
        "preds": all_preds,
        "labels": all_labels,
        "confidences": all_confs,
        "report": report,
        "confusion_matrix": cm,
    }

In [21]:
def summarize_unlabeled_predictions(model, loader, class_names, device, run_name="model"):
    model.eval()

    pred_indices = []
    pred_confidences = []

    with torch.no_grad():
        for x in tqdm(loader, desc=f"Inferring unlabeled with {run_name}"):
            x = x.to(device)

            logits = model(x)
            probs = torch.softmax(logits, dim=1)

            confs, preds = probs.max(dim=1)

            pred_indices.extend(preds.cpu().numpy())
            pred_confidences.extend(confs.cpu().numpy())

    pred_indices = np.array(pred_indices)
    pred_confidences = np.array(pred_confidences)

    total = len(pred_indices)
    rows = []

    for class_idx, class_name in enumerate(class_names):
        mask = pred_indices == class_idx
        count = mask.sum()
        pct = (count / total * 100) if total > 0 else 0.0

        if count > 0:
            conf_vals = pred_confidences[mask]
            rows.append({
                "class": class_name,
                "count": int(count),
                "percent": round(pct, 2),
                "mean_conf": round(float(conf_vals.mean()), 4),
                "median_conf": round(float(np.median(conf_vals)), 4),
                "q25_conf": round(float(np.percentile(conf_vals, 25)), 4),
                "q75_conf": round(float(np.percentile(conf_vals, 75)), 4),
                "min_conf": round(float(conf_vals.min()), 4),
                "max_conf": round(float(conf_vals.max()), 4),
            })
        else:
            rows.append({
                "class": class_name,
                "count": 0,
                "percent": 0.0,
                "mean_conf": np.nan,
                "median_conf": np.nan,
                "q25_conf": np.nan,
                "q75_conf": np.nan,
                "min_conf": np.nan,
                "max_conf": np.nan,
            })

    summary_df = pd.DataFrame(rows).sort_values("count", ascending=False).reset_index(drop=True)

    print(f"\n--- Unlabeled prediction summary: {run_name} ---")
    print(f"Total unlabeled samples: {total:,}\n")
    print(summary_df.to_string(index=False))

    return summary_df

In [22]:
def _synchronize_device(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elif device.type == "mps":
        try:
            torch.mps.synchronize()
        except Exception:
            pass

def benchmark_inference_time(model, loader, device, warmup_batches=5, max_batches=None):
    model = model.to(device)
    model.eval()

    total_samples = 0
    total_time = 0.0
    measured_batches = 0

    with torch.no_grad():
        # warmup
        for i, batch in enumerate(loader):
            x = batch[0] if isinstance(batch, (list, tuple)) else batch
            x = x.to(device)

            _ = model(x)
            _synchronize_device(device)

            if i + 1 >= warmup_batches:
                break

        # timed pass
        for i, batch in enumerate(loader):
            if max_batches is not None and i >= max_batches:
                break

            x = batch[0] if isinstance(batch, (list, tuple)) else batch
            x = x.to(device)

            _synchronize_device(device)
            start = time.perf_counter()

            _ = model(x)

            _synchronize_device(device)
            end = time.perf_counter()

            batch_time = end - start
            batch_size = x.shape[0]

            total_time += batch_time
            total_samples += batch_size
            measured_batches += 1

    avg_batch_time_ms = (total_time / measured_batches) * 1000 if measured_batches > 0 else np.nan
    avg_sample_time_ms = (total_time / total_samples) * 1000 if total_samples > 0 else np.nan
    throughput_samples_sec = total_samples / total_time if total_time > 0 else np.nan

    return {
        "device": str(device),
        "batches_measured": measured_batches,
        "samples_measured": total_samples,
        "avg_batch_time_ms": avg_batch_time_ms,
        "avg_sample_time_ms": avg_sample_time_ms,
        "throughput_samples_sec": throughput_samples_sec,
    }

def benchmark_on_available_devices(model_class, model_state_dict, loader, num_classes):
    benchmark_rows = []

    candidate_devices = [torch.device("cpu")]
    if torch.cuda.is_available():
        candidate_devices.append(torch.device("cuda"))
    elif torch.backends.mps.is_available():
        candidate_devices.append(torch.device("mps"))

    for dev in candidate_devices:
        model = model_class(num_classes=num_classes)
        model.load_state_dict(model_state_dict)
        model.to(dev)

        stats = benchmark_inference_time(
            model=model,
            loader=loader,
            device=dev,
            warmup_batches=5,
            max_batches=50,  # keep timing practical
        )
        benchmark_rows.append(stats)

    return pd.DataFrame(benchmark_rows)

### CNNLarge

Image Input size = 64x64

In [23]:
class CNNLarge(nn.Module):
    def __init__(self, num_classes=9):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 1x64x64 -> 32x32x32
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2), # 64 -> 32
            
            # Block 2: 32x32x32 -> 64x16x16
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2), # 32 -> 16
            
            # Block 3: 64x16x16 -> 128x8x8
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2), # 16 -> 8
        )
        # 128 channels * 8*8 spatial = 8192
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(128 * 8 * 8, 256)
        self.head = nn.Linear(256, num_classes)
    
    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        x = self.dropout(x)
        x = torch.relu(self.fc(x))
        return self.head(x)

cnn_large = CNNLarge().to(device)
total_params = sum(p.numel() for p in cnn_large.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

Total trainable parameters: 2,387,049


In [25]:
model_path = "checkpoints/best/cnn_large.pt"

cnn_large = CNNLarge(num_classes=NUM_CLASSES).to(device)

if os.path.exists(model_path):
    ckpt = torch.load(model_path, map_location=device, weights_only=False)
    cnn_large.load_state_dict(ckpt["model_state_dict"])
    training_metadata = ckpt.get("results", None) if "ckpt" in locals() else None
    print(f"Loaded {model_path}")
else:
    raise FileNotFoundError(f"No checkpoint found at {model_path}")

Loaded checkpoints/best/cnn_large.pt


In [26]:
cnnlarge_test_results = evaluate_classifier(
    model=cnn_large,
    loader=test_loader,
    class_names=CLASSES,
    device=device,
    run_name="cnn_large",
    label_smoothing=training_metadata["hyperparams"]["label_smoothing"],
    training_metadata=training_metadata,
)

Evaluating cnn_large:   0%|          | 0/136 [00:00<?, ?it/s]


--- cnn_large ---
Test Loss: 0.5317

Per-class Metrics:
Class                F1  Precision     Recall    Mean Conf
------------------------------------------------------------
none             0.9918     0.9881     0.9957       0.9031
Edge-Ring        0.9810     0.9765     0.9855       0.9064
Edge-Loc         0.8557     0.8848     0.8285       0.8122
Center           0.9476     0.9583     0.9371       0.8928
Loc              0.7872     0.8982     0.7006       0.7718
Scratch          0.8504     0.8690     0.8326       0.8148
Random           0.9169     0.9091     0.9249       0.8416
Donut            0.8889     0.8772     0.9009       0.8129
Near-full        0.9355     0.9062     0.9667       0.8653
------------------------------------------------------------
macro avg        0.9061     0.9186     0.8969       0.8971

Overall Accuracy: 0.9807
Training Time: 12.5120 min
Best Val Loss at F1: 0.5318
Best Train Loss at F1: 0.5384


In [27]:
cnnlarge_unlabeled_summary = summarize_unlabeled_predictions(
    model=cnn_large,
    loader=unlabeled_loader,
    class_names=CLASSES,
    device=device,
    run_name="cnn_large",
)

Inferring unlabeled with cnn_large:   0%|          | 0/2495 [00:00<?, ?it/s]


--- Unlabeled prediction summary: cnn_large ---
Total unlabeled samples: 638,507

    class  count  percent  mean_conf  median_conf  q25_conf  q75_conf  min_conf  max_conf
     none 517615    81.07     0.8150       0.8819    0.7399    0.9080    0.1914    0.9750
      Loc  47893     7.50     0.5216       0.4945    0.4533    0.5775    0.1904    0.9932
 Edge-Loc  26848     4.20     0.6486       0.6480    0.4521    0.8496    0.2022    0.9944
   Center  17145     2.69     0.8000       0.8874    0.6932    0.9375    0.2021    0.9888
Edge-Ring  11644     1.82     0.7328       0.8015    0.5508    0.9226    0.2015    0.9820
   Random   7332     1.15     0.7983       0.8769    0.6910    0.9395    0.1896    0.9892
  Scratch   7203     1.13     0.5300       0.4744    0.4033    0.6296    0.2020    0.9874
Near-full   2352     0.37     0.6712       0.5629    0.4916    0.9044    0.3227    0.9726
    Donut    475     0.07     0.6556       0.6642    0.4999    0.8337    0.2114    0.9653


In [28]:
cnnlarge_timing_df = benchmark_on_available_devices(
    model_class=CNNLarge,
    model_state_dict=cnn_large.state_dict(),
    loader=unlabeled_loader,
    num_classes=NUM_CLASSES,
)

print(cnnlarge_timing_df.to_string(index=False))

device  batches_measured  samples_measured  avg_batch_time_ms  avg_sample_time_ms  throughput_samples_sec
   cpu                50             12800         387.098408            1.512103              661.330542
   mps                50             12800          33.444173            0.130641             7654.547290


### CNNSmall

Image Input size = 64x64

In [29]:
class CNNSmall(nn.Module):
    def __init__(self, num_classes=9):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 1x64x64 -> 16x32x32
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
            nn.Conv2d(16, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
            nn.MaxPool2d(2), # 64 -> 32

            # Block 2: 16x32x32 -> 32x16x16
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2), # 32 -> 16

            # Block 3: 32x16x16 -> 64x8x8
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2), # 16 -> 8
        )
        # 64 channels * 8*8 spatial = 4096
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(64 * 8 * 8, 128)
        self.head = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        x = self.dropout(x)
        x = torch.relu(self.fc(x))
        return self.head(x)
        
cnn_small = CNNSmall().to(device)
total_params = sum(p.numel() for p in cnn_small.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

Total trainable parameters: 597,817


In [30]:
model_path = "checkpoints/best/cnn_small.pt"

cnn_small = CNNSmall(num_classes=NUM_CLASSES).to(device)

if os.path.exists(model_path):
    ckpt = torch.load(model_path, map_location=device, weights_only=False)
    cnn_small.load_state_dict(ckpt["model_state_dict"])
    training_metadata = ckpt.get("results", None) if "ckpt" in locals() else None
    print(f"Loaded {model_path}")
else:
    raise FileNotFoundError(f"No checkpoint found at {model_path}")

Loaded checkpoints/best/cnn_small.pt


In [227]:
cnnsmall_test_results = evaluate_classifier(
    model=cnn_small,
    loader=test_loader,
    class_names=CLASSES,
    device=device,
    run_name="cnn_small",
    label_smoothing=training_metadata["hyperparams"]["label_smoothing"],
    training_metadata=training_metadata,
)

Evaluating cnn_small:   0%|          | 0/136 [00:00<?, ?it/s]


--- cnn_small ---
Test Loss: 0.5381

Per-class Metrics:
Class                F1  Precision     Recall    Mean Conf
------------------------------------------------------------
none             0.9906     0.9852     0.9961       0.9029
Edge-Ring        0.9767     0.9695     0.9840       0.8999
Edge-Loc         0.8309     0.9204     0.7572       0.7644
Center           0.9324     0.9675     0.8999       0.8739
Loc              0.7701     0.8743     0.6880       0.7556
Scratch          0.7800     0.7471     0.8159       0.7767
Random           0.9054     0.8977     0.9133       0.8192
Donut            0.8487     0.7953     0.9099       0.8351
Near-full        0.9355     0.9062     0.9667       0.8201
------------------------------------------------------------
macro avg        0.8856     0.8959     0.8812       0.8945

Overall Accuracy: 0.9775
Training Time: 8.9769 min
Best Val Loss at F1: 0.5381
Best Train Loss at F1: 0.5582


In [228]:
cnnsmall_unlabeled_summary = summarize_unlabeled_predictions(
    model=cnn_small,
    loader=unlabeled_loader,
    class_names=CLASSES,
    device=device,
    run_name="cnn_small",
)

Inferring unlabeled with cnn_small:   0%|          | 0/2495 [00:00<?, ?it/s]


--- Unlabeled prediction summary: cnn_small ---
Total unlabeled samples: 638,507

    class  count  percent  mean_conf  median_conf  q25_conf  q75_conf  min_conf  max_conf
     none 527470    82.61     0.8176       0.8804    0.7547    0.9109    0.1908    0.9815
      Loc  39648     6.21     0.5093       0.5085    0.4651    0.5375    0.2022    0.9926
 Edge-Loc  19090     2.99     0.6191       0.6131    0.4610    0.7825    0.2012    0.9942
  Scratch  14484     2.27     0.5033       0.4700    0.3991    0.5783    0.2114    0.9973
   Center  14122     2.21     0.7730       0.8555    0.6382    0.9268    0.2148    0.9880
Edge-Ring  12956     2.03     0.6972       0.7284    0.5076    0.9103    0.1929    0.9901
   Random   7799     1.22     0.7213       0.7802    0.5346    0.9179    0.2005    0.9822
Near-full   2288     0.36     0.8473       0.8771    0.8684    0.8898    0.2157    0.9477
    Donut    650     0.10     0.6766       0.6869    0.4891    0.8822    0.2243    0.9845


In [229]:
cnnsmall_timing_df = benchmark_on_available_devices(
    model_class=CNNSmall,
    model_state_dict=cnn_small.state_dict(),
    loader=unlabeled_loader,
    num_classes=NUM_CLASSES,
)

print(cnnsmall_timing_df.to_string(index=False))

device  batches_measured  samples_measured  avg_batch_time_ms  avg_sample_time_ms  throughput_samples_sec
   cpu                50             12800         119.144940            0.465410             2148.643492
   mps                50             12800          13.871142            0.054184            18455.582480


### ResnNet18

In [31]:
from torchvision.models import resnet18

class ResNet18(nn.Module):
    def __init__(self, num_classes=9):
        super().__init__()
        self.backbone = resnet18(weights=None)
        self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, num_classes)
    def forward(self, x):
        return self.backbone(x)

resnet = ResNet18(num_classes=9).to(device)
total_params = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

Total trainable parameters: 11,174,857


In [32]:
model_path = "checkpoints/best/resnet.pt"

resnet = ResNet18(num_classes=NUM_CLASSES).to(device)

if os.path.exists(model_path):
    ckpt = torch.load(model_path, map_location=device, weights_only=False)
    resnet.load_state_dict(ckpt["model_state_dict"])
    training_metadata = ckpt.get("results", None) if "ckpt" in locals() else None
    print(f"Loaded {model_path}")
else:
    raise FileNotFoundError(f"No checkpoint found at {model_path}")

Loaded checkpoints/best/resnet.pt


In [232]:
resnet_test_results = evaluate_classifier(
    model=resnet,
    loader=test_loader,
    class_names=CLASSES,
    device=device,
    run_name="resnet",
    label_smoothing=training_metadata["hyperparams"]["label_smoothing"],
    training_metadata=training_metadata,
)

Evaluating resnet:   0%|          | 0/136 [00:00<?, ?it/s]


--- resnet ---
Test Loss: 0.5455

Per-class Metrics:
Class                F1  Precision     Recall    Mean Conf
------------------------------------------------------------
none             0.9896     0.9861     0.9931       0.8976
Edge-Ring        0.9734     0.9741     0.9726       0.8947
Edge-Loc         0.8147     0.8558     0.7775       0.8074
Center           0.9344     0.9317     0.9371       0.8451
Loc              0.7365     0.7961     0.6852       0.7711
Scratch          0.6572     0.7554     0.5816       0.7831
Random           0.8914     0.8602     0.9249       0.8356
Donut            0.8245     0.7537     0.9099       0.8093
Near-full        0.9355     0.9062     0.9667       0.8814
------------------------------------------------------------
macro avg        0.8619     0.8688     0.8610       0.8901

Overall Accuracy: 0.9742
Training Time: 5.4844 min
Best Val Loss at F1: 0.5461
Best Train Loss at F1: 0.5408


In [233]:
resnet_timing_df = benchmark_on_available_devices(
    model_class=ResNet18,
    model_state_dict=resnet.state_dict(),
    loader=unlabeled_loader,
    num_classes=NUM_CLASSES,
)

print(resnet_timing_df.to_string(index=False))

device  batches_measured  samples_measured  avg_batch_time_ms  avg_sample_time_ms  throughput_samples_sec
   cpu                50             12800         511.457755            1.997882              500.530097
   mps                50             12800          32.133718            0.125522             7966.709530


### Autoencoder Classifier

Image Input size = 64x64

In [33]:
import copy

class AE_Large(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.encoder_dropout = nn.Dropout(0.2)
        self.encoder_fc = nn.Linear(128 * 8 * 8, latent_dim)
        self.decoder_fc = nn.Linear(latent_dim, 128 * 8 * 8)
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(128, 128, 4, stride=2, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.ConvTranspose2d(64, 64, 4, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def encode(self, x):
        x = self.encoder_conv(x)
        x = x.flatten(1)
        x = self.encoder_dropout(x)
        return self.encoder_fc(x)

    def decode(self, z):
        x = torch.relu(self.decoder_fc(z))
        x = x.view(-1, 128, 8, 8)
        return self.decoder_conv(x)

    def forward(self, x):
        return self.decode(self.encode(x))


class AEClassifier(nn.Module):
    def __init__(self, encoder, latent_dim=128, num_classes=9, freeze_encoder=True):
        super().__init__()
        self.encoder = encoder
        self.freeze_encoder = freeze_encoder
        for param in self.encoder.parameters():
            param.requires_grad = not freeze_encoder
        self.head = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        if self.freeze_encoder:
            with torch.no_grad():
                z = self.encoder.encode(x)
        else:
            z = self.encoder.encode(x)
        return self.head(z)


class AEClassifierBench(nn.Module):
    def __init__(self, num_classes=9, latent_dim=128):
        super().__init__()
        self.encoder = AE_Large(latent_dim=latent_dim)
        self.freeze_encoder = False
        self.head = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        z = self.encoder.encode(x)
        return self.head(z)


encoder_shell = AE_Large(latent_dim=128)
ae_classifier = AEClassifier(encoder_shell, latent_dim=128, num_classes=NUM_CLASSES, freeze_encoder=False).to(device)
total_params = sum(p.numel() for p in ae_classifier.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

Total trainable parameters: 2,831,690


In [34]:
model_path = "checkpoints/best/autoenc_classifier_dunlabeled.pt"

encoder_shell = AE_Large(latent_dim=128)
ae_classifier = AEClassifier(encoder_shell, latent_dim=128, num_classes=NUM_CLASSES, freeze_encoder=False).to(device)

if os.path.exists(model_path):
    ckpt = torch.load(model_path, map_location=device, weights_only=False)
    ae_classifier.load_state_dict(ckpt["model_state_dict"])
    training_metadata = ckpt.get("results", None) if "ckpt" in locals() else None
    print(f"Loaded {model_path}")
else:
    raise FileNotFoundError(f"No checkpoint found at {model_path}")

Loaded checkpoints/best/autoenc_classifier_dunlabeled.pt


In [236]:
aeclassifier_test_results = evaluate_classifier(
    model=ae_classifier,
    loader=test_loader,
    class_names=CLASSES,
    device=device,
    run_name="ae_classifier",
    label_smoothing=0.1,
    training_metadata=training_metadata,
)

Evaluating ae_classifier:   0%|          | 0/136 [00:00<?, ?it/s]


--- ae_classifier ---
Test Loss: 1.6075

Per-class Metrics:
Class                F1  Precision     Recall    Mean Conf
------------------------------------------------------------
none             0.9895     0.9851     0.9940       0.9905
Edge-Ring        0.9754     0.9772     0.9737       0.9851
Edge-Loc         0.8275     0.8574     0.7996       0.8857
Center           0.9383     0.9467     0.9302       0.9628
Loc              0.7086     0.7783     0.6504       0.8039
Scratch          0.4697     0.5575     0.4059       0.6851
Random           0.9112     0.9034     0.9191       0.9495
Donut            0.8297     0.8051     0.8559       0.9259
Near-full        0.9333     0.9333     0.9333       0.9737
------------------------------------------------------------
macro avg        0.8426     0.8605     0.8291       0.9814

Overall Accuracy: 0.9733
Training Time: 12.3440 min


In [237]:
aeclassifier_timing_df = benchmark_on_available_devices(
    model_class=AEClassifierBench,
    model_state_dict=ae_classifier.state_dict(),
    loader=unlabeled_loader,
    num_classes=NUM_CLASSES,
)

print(aeclassifier_timing_df.to_string(index=False))

device  batches_measured  samples_measured  avg_batch_time_ms  avg_sample_time_ms  throughput_samples_sec
   cpu                50             12800         277.213302            1.082864              923.476610
   mps                50             12800          33.077809            0.129210             7739.327568
